# QazRuFake v3: protocol rows for XLM-R and a zero-shot LLM baseline

1. *Runtime → Change runtime type → **T4 GPU***.
2. Drag **`data/qazrufake.jsonl`** into the Files panel on the left (📁) and wait for the upload to finish.
3. *Runtime → **Run all***. Allow Google Drive access when asked.

Part A trains XLM-R base (3 seeds) on the Russian fact-checking outlets only, on original and claim-only text, to fill the two missing Russian bars of Figure 1 (~15 min).
Part B runs a zero-shot instruction-tuned LLM (default: Qwen2.5-7B-Instruct, 4-bit) on every Russian and Kazakh test item, original and claim-only text (2,162 prompts, ~30–40 min). No training, no dev data.

Both parts skip work that is already saved, so after a disconnect just press *Run all* again.
When it finishes, send back from Google Drive: **`qazrufake_preds_v3.jsonl`**, **`qazrufake_preds_llm.jsonl`** and **`qazrufake_v3_summary.json`**.


In [ ]:
# 1. GPU check, packages, Drive, corpus
import torch, os, json
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch Runtime -> T4 GPU, then rerun')
!pip install -q transformers datasets accelerate scikit-learn bitsandbytes
from google.colab import drive
drive.mount('/content/drive')
CORPUS = next((p for p in ['qazrufake.jsonl', 'qazrufake_full.jsonl'] if os.path.exists(p)), None)
assert CORPUS, 'Upload data/qazrufake.jsonl into the Files panel (left), then run again'
R = [json.loads(l) for l in open(CORPUS, encoding='utf-8')]
assert all(r.get('text') for r in R), 'Some items have no text'
D = '/content/drive/MyDrive/'
OUT = D + 'qazrufake_preds_v3.jsonl'
OUT_LLM = D + 'qazrufake_preds_llm.jsonl'
print(len(R), 'items')

## Part A. XLM-R trained and tested on the Russian fact-checking outlets only

In [ ]:
# 2. Settings: Russian fact-checking outlets (Tengrinews excluded from train, dev and test)
rufc = lambda r: r['lang'] == 'ru' and r['source_name'] != 'Tengrinews.kz'
SETTINGS = {'rufc|text': (rufc, 'text'), 'rufc|claim': (rufc, 'claim')}
SEEDS = [1, 2, 3]
MODEL = 'xlm-roberta-base'

In [ ]:
# 3. Training function: trains one setting/seed and writes a prediction for every test item (RU and KK)
import shutil, numpy as np
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

def done(setting, seed):
    if not os.path.exists(OUT): return False
    key = f'"setting": "{setting}", "seed": {seed}'
    return any(key in l for l in open(OUT))

def run(setting, seed, epochs=4, lr=2e-5, bs=16, maxlen=256):
    if done(setting, seed):
        print('skip (already done):', setting, seed); return
    filt, field = SETTINGS[setting]
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL)
    def ds(rs):
        d = Dataset.from_dict({'text': [r[field] for r in rs], 'label': [r['label'] for r in rs]})
        return d.map(lambda b: tok(b['text'], truncation=True, max_length=maxlen), batched=True)
    tr = [r for r in R if r['split'] == 'train' and filt(r)]
    dv = [r for r in R if r['split'] == 'dev' and filt(r)]
    te = [r for r in R if r['split'] == 'test']
    dtr, ddv = ds(tr), ds(dv)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    def metrics(p):
        y = p.label_ids; pr = p.predictions.argmax(-1)
        return {'f1m': f1_score(y, pr, average='macro'), 'acc': accuracy_score(y, pr)}
    args = TrainingArguments(output_dir='out', learning_rate=lr, per_device_train_batch_size=bs,
        per_device_eval_batch_size=64, num_train_epochs=epochs, weight_decay=0.01,
        warmup_steps=int(0.1 * epochs * -(-len(dtr) // bs)),
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='f1m', save_total_limit=1, fp16=torch.cuda.is_available(),
        report_to='none', seed=seed)
    t = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=ddv,
                data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
    t.train()
    p = t.predict(ds(te))
    prob = torch.softmax(torch.tensor(p.predictions), -1)[:, 1].numpy()
    with open(OUT, 'a') as f:
        for r, q in zip(te, prob):
            f.write(json.dumps({'model': MODEL, 'setting': setting, 'seed': seed, 'uid': r['uid'],
                                'test': r['lang'], 'prob': round(float(q), 5)}) + '\n')
    y = np.array([r['label'] for r in te]); pr = (prob > 0.5).astype(int)
    for lang in ['ru', 'kk']:
        m = np.array([r['lang'] == lang for r in te])
        print(setting, seed, lang, 'F1m = %.3f' % f1_score(y[m], pr[m], average='macro'))
    del t, model; torch.cuda.empty_cache(); shutil.rmtree('out', ignore_errors=True)

In [ ]:
# 4. Run Part A
for s in SETTINGS:
    for seed in SEEDS:
        print('=' * 20, s, seed)
        run(s, seed)
print('PART A DONE')

## Part B. Zero-shot LLM

In [ ]:
# 5. Load the LLM in 4-bit (change LLM to another instruct model if you prefer)
import gc; gc.collect(); torch.cuda.empty_cache()
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLM = 'Qwen/Qwen2.5-7B-Instruct'
ltok = AutoTokenizer.from_pretrained(LLM, padding_side='left')
llm = AutoModelForCausalLM.from_pretrained(LLM, device_map='auto',
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                               bnb_4bit_quant_type='nf4'))
llm.eval()
LANG = {'ru': 'Russian', 'kk': 'Kazakh'}
SYSTEM = 'You are a careful fact-checker for news and social media in Kazakhstan.'
def prompt(text, lang):
    user = (f'The following {LANG[lang]}-language text reports a claim or a piece of news from Kazakhstan.\n'
            'Decide whether what it reports is true (REAL) or false or misleading (FAKE).\n'
            'Answer with exactly one word: REAL or FAKE.\n\nText:\n' + text)
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}]
    return ltok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ID_REAL = ltok.encode('REAL', add_special_tokens=False)[0]
ID_FAKE = ltok.encode('FAKE', add_special_tokens=False)[0]
assert ID_REAL != ID_FAKE
print('answer tokens', ltok.decode([ID_REAL]), ltok.decode([ID_FAKE]))

In [ ]:
# 6. Score every test item, original and claim-only text. P(FAKE) = softmax over the two answer tokens.
done_llm = set()
if os.path.exists(OUT_LLM):
    done_llm = {(d['field'], d['uid']) for d in map(json.loads, open(OUT_LLM))}
test = [r for r in R if r['split'] == 'test']
jobs = [(f, r) for f in ['text', 'claim'] for r in test if (f, r['uid']) not in done_llm]
print(len(jobs), 'prompts to run')
BS = 8
with torch.no_grad():
    for i in range(0, len(jobs), BS):
        batch = jobs[i:i + BS]
        enc = ltok([prompt(r[f][:2000], r['lang']) for f, r in batch], return_tensors='pt', padding=True).to(llm.device)
        logits = llm(**enc).logits[:, -1, :].float()
        two = torch.softmax(logits[:, [ID_REAL, ID_FAKE]], -1)[:, 1].cpu().numpy()
        with open(OUT_LLM, 'a') as fo:
            for (f, r), p in zip(batch, two):
                fo.write(json.dumps({'model': LLM, 'field': f, 'uid': r['uid'], 'prob': round(float(p), 5)}) + '\n')
        if (i // BS) % 25 == 0: print(i + len(batch), '/', len(jobs))
print('PART B DONE')

In [ ]:
# 7. Summary on the protocol test sets
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
by = {r['uid']: r for r in R}
SUBSETS = {
    'ru|all':  lambda r: r['lang'] == 'ru',
    'ru|fcout': lambda r: r['lang'] == 'ru' and r['source_name'] != 'Tengrinews.kz',
    'kk|all':  lambda r: r['lang'] == 'kk',
    'kk|fc':   lambda r: r['lang'] == 'kk' and r['fc_kk'],
    'kk|unamb': lambda r: r['lang'] == 'kk' and r['unamb'],
}
def score(df):
    y = df.uid.map(lambda u: by[u]['label']).values; p = df.prob.values
    return f1_score(y, (p > 0.5).astype(int), average='macro'), (roc_auc_score(y, p) if len(set(y)) > 1 else None), len(df)
summary = {}
L = pd.DataFrame(map(json.loads, open(OUT_LLM)))
for f in ['text', 'claim']:
    for sname, filt in SUBSETS.items():
        d = L[(L.field == f) & L.uid.map(lambda u: filt(by[u]))]
        f1, auc, n = score(d)
        summary[f'llm|{f}|{sname}'] = {'f1m': f1, 'auc': auc, 'n': n, 'pred_fake_rate': float((d.prob > 0.5).mean())}
X = pd.DataFrame(map(json.loads, open(OUT)))
for s in SETTINGS:
    vals = []
    for seed in SEEDS:
        d = X[(X.setting == s) & (X.seed == seed) & X.uid.map(lambda u: SUBSETS['ru|fcout'](by[u]))]
        vals.append(score(d)[0])
    summary[f'xlmr|{s}|ru|fcout'] = {'f1_mean': float(np.mean(vals)), 'f1_sd': float(np.std(vals, ddof=1)), 'n': int(len(d))}
json.dump(summary, open(D + 'qazrufake_v3_summary.json', 'w'), indent=1)
print(pd.DataFrame(summary).T.round(3).to_string())